<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week7/Day3/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Créer un système de génération augmentée pour la récupération (RAG)

## Étape 1 : Configurer votre environnement

Nous allons installer toutes les bibliothèques requises pour construire le système RAG.

In [ ]:
# Installez toutes les bibliothèques requises
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -U langchain-community

## Étape 2 : Chargement du jeu de données

Nous allons charger le jeu de données `databricks/databricks-dolly-15k` à l'aide de `HuggingFaceDatasetLoader`.

In [ ]:
# Importez HuggingFaceDatasetLoader
from langchain_community.document_loaders import HuggingFaceDatasetLoader

# Spécifiez le nom de l'ensemble de données et la colonne de contenu
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

# Créez une instance HuggingFaceDatasetLoader et chargez les données sous forme de documents
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# Facultatif : Affichez les 2 premières entrées pour vérifier le chargement
print(data[:2])

[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


In [ ]:
# Afficher les 2 premières entrées pour vérifier le chargement
print(data[:2])

## Étape 3 : Fractionner les documents

Nous allons diviser les documents en morceaux plus petits et qui se chevauchent à l'aide de `RecursiveCharacterTextSplitter`.

In [ ]:
# Importez RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Créez une instance RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Fractionner les documents chargés
docs = text_splitter.split_documents(data)

# Facultatif : Affichez le premier morceau de document
print(docs[0])

page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## Étape 4 : Intégration du texte

Nous allons créer des embeddings pour le texte à l'aide de `HuggingFaceEmbeddings`.

In [ ]:
# Importez HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Définissez le chemin du modèle, les configurations du modèle et les options d'encodage
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

# Initialisation HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

# (Facultatif) Création de l'intégration de test
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[-0.038338519632816315, 0.1234646886587143, -0.028642984107136726]


## Étape 5 : Créer un entrepôt de vecteurs

Nous allons créer un entrepôt de vecteurs FAISS à partir des fragments de document et des intégrations.

In [ ]:
# Importez FAISS
from langchain_community.vectorstores import FAISS

# Créez un espace de stockage vectoriel FAISS
db = FAISS.from_documents(docs, embeddings)

print("FAISS vector store created.")

FAISS vector store created.


## Étape 6 : Préparation du modèle LLM

Nous allons charger un modèle de langage pré-entraîné (`Intel/dynamic_tinybert`) pour la réponse aux questions.

In [ ]:
# Importez les classes nécessaires depuis transformers et langchain
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

# Charger le tokenizer et le modèle de question-réponse
model_name = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Créer un pipeline de questions-réponses
qa_pipeline = pipeline(
  "question-answering",
  model=model,
  tokenizer=tokenizer,
  return_tensors='pt'
)

# Créer un wrapper de pipeline Langchain
llm = HuggingFacePipeline(
  pipeline=qa_pipeline,
  model_kwargs={"temperature": 0.7, "max_length": 512}
)

Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key              | Status     |  | 
-----------------+------------+--+-
fit_dense.weight | UNEXPECTED |  | 
fit_dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

## Étape 7 : Création de la chaîne de questions-réponses pour la recherche

Nous allons construire la chaîne `RetrievalQA` pour connecter le module de recherche et le LLM.

In [ ]:
# Importez RetrievalQA
from langchain_community.chains import RetrievalQA

# Créez un récupérateur à partir de votre base de données FAISS
retriever = db.as_retriever(search_kwargs={"k": 4}) # k est le nombre de documents à récupérer

# Constituez la chaîne RetrievalQA
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="refine", retriever=retriever, return_source_documents=False)

print("RetrievalQA chain created.")

## Étape 8 : Testez votre système RAG

Enfin, nous allons exécuter une requête de test pour vérifier le fonctionnement du système RAG.

In [ ]:
# Définissez votre question
question = "What is cheesemaking?"

# Exécutez la chaîne d'assurance qualité et affichez le résultat
result = qa.run({"query": question})
print(result)